In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from keras import layers, Model
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.optimize import differential_evolution
import warnings
warnings.filterwarnings('ignore')

In [2]:
class EnhancedGRANXModel(tf.keras.Model):
    def __init__(self, sequence_length, n_features, hidden_units=64, attention_heads=8, 
                 correction_rate=0.01, dropout_rate=0.2, n_external_features=0):
        super(EnhancedGRANXModel, self).__init__()
        
        self.sequence_length = sequence_length
        self.n_features = n_features
        self.hidden_units = hidden_units
        self.correction_rate = correction_rate
        
        # Multi-scale CNN layers
        self.conv_layers = [
            layers.Conv1D(32, 3, activation='relu', padding='same'),
            layers.BatchNormalization(),
            layers.Conv1D(64, 5, activation='relu', padding='same'),
            layers.BatchNormalization(),
            layers.Dropout(dropout_rate)
        ]
        
        # Hierarchical temporal modeling
        self.gru_layer = layers.GRU(hidden_units, return_sequences=True, dropout=dropout_rate)
        self.lstm_layer = layers.LSTM(hidden_units, return_sequences=True, dropout=dropout_rate)
        self.temporal_fusion = layers.Dense(hidden_units, activation='tanh')
        
        # Enhanced attention mechanism
        self.multi_head_attention = layers.MultiHeadAttention(
            num_heads=attention_heads, key_dim=hidden_units, dropout=dropout_rate
        )
        
        # External feature processing
        if n_external_features > 0:
            self.external_processor = layers.Dense(hidden_units//2, activation='relu')
        else:
            self.external_processor = None
        
        # Feature importance layers
        self.feature_importance = layers.Dense(hidden_units, activation='sigmoid')
        
        # Gradient correction mechanism
        self.gradient_corrector = layers.Dense(hidden_units, activation='tanh')
        
        # Global pooling and output
        self.global_pool = layers.GlobalAveragePooling1D()
        self.feature_dropout = layers.Dropout(dropout_rate)
        self.output_dropout = layers.Dropout(dropout_rate)
        
        # Multi-layer output projection
        self.output_layers = [
            layers.Dense(hidden_units//2, activation='relu'),
            layers.Dense(hidden_units//4, activation='relu'),
            layers.Dense(1, activation='linear')
        ]
        
    def compute_hierarchical_features(self, x):
        conv_outputs = []
        for layer in self.conv1d_layers:
            x = layer(x)
            if isinstance(layer, layers.Conv1D):
                conv_outputs.append(x)
        
        if len(conv_outputs) > 1:
            x = layers.Concatenate(axis=-1)(conv_outputs[-2:])
        
        for layer in self.temporal_layers:
            x = layer(x)
        
        for dense in self.feature_dense:
            x = dense(x)
        M_t = self.feature_norm(x)
        
        return M_t
    
    def compute_multi_scale_attention(self, M_t, external_features=None):
        attended_self, self_weights = self.attention(
            M_t, M_t, return_attention_scores=True
        )
        
        if external_features is not None:
            attended_cross, cross_weights = self.cross_attention(
                M_t, external_features, return_attention_scores=True
            )
            attended_features = layers.Add()([attended_self, attended_cross])
            combined_weights = layers.Concatenate()([self_weights, cross_weights])
            alpha_t = tf.nn.softmax(tf.reduce_mean(combined_weights, axis=[1, 2]), axis=-1)
        else:
            attended_features = attended_self
            alpha_t = tf.nn.softmax(tf.reduce_mean(self_weights, axis=1), axis=-1)
        
        return attended_features, alpha_t
    
    def compute_adaptive_correction(self, weighted_features, iteration=0):
        adaptive_rate = self.correction_rate * (1 / (1 + 0.01 * iteration))
        
        correction = weighted_features
        for dense in self.correction_dense:
            correction = dense(correction)
        
        correction = -adaptive_rate * correction
        
        return correction
    
    def call(self, inputs, training=None, external_features=None, iteration=0, **kwargs):
        batch_size = tf.shape(inputs)[0]
        seq_len = tf.shape(inputs)[1]
        
        # Multi-scale CNN feature extraction
        x = inputs
        for conv_layer in self.conv_layers:
            x = conv_layer(x)
        
        # Hierarchical temporal modeling
        gru_out = self.gru_layer(x)
        lstm_out = self.lstm_layer(x)
        
        temporal_combined = tf.concat([gru_out, lstm_out], axis=-1)
        temporal_features = self.temporal_fusion(temporal_combined)
        
        # Multi-head attention
        attended_features, _ = self.multi_head_attention(
            temporal_features, temporal_features, return_attention_scores=True
        )
        
        # External feature integration
        if external_features is not None and self.external_processor is not None:
            external_processed = self.external_processor(external_features)
            external_expanded = tf.expand_dims(external_processed, axis=1)
            external_tiled = tf.tile(external_expanded, [1, seq_len, 1])
            attended_features = tf.concat([attended_features, external_tiled], axis=-1)
        
        # Feature importance computation
        importance_scores = self.feature_importance(attended_features)
        weighted_features = attended_features * importance_scores
        
        # Adaptive gradient correction
        if training and iteration > 0:
            correction = self.gradient_corrector(weighted_features)
            correction = self.correction_rate * correction
            corrected_features = weighted_features - correction
        else:
            corrected_features = weighted_features
        
        # Global feature aggregation and output
        pooled_features = self.global_pool(corrected_features)
        pooled_features = self.feature_dropout(pooled_features, training=training)
        
        for i, dense_layer in enumerate(self.output_layers):
            pooled_features = dense_layer(pooled_features)
            if i < len(self.output_layers) - 1:
                pooled_features = self.output_dropout(pooled_features, training=training)
        
        return pooled_features

In [3]:
class ApplianceOptimizer:
    def __init__(self, model, tariff_schedule, appliance_profiles):
        self.model = model
        self.tariff_schedule = tariff_schedule
        self.appliance_profiles = appliance_profiles
        self.optimization_horizon = 24
        
    def get_appliance_constraints(self, appliance):
        constraints = {
            'washing_machine': {'duration': 2, 'window': (6, 22), 'priority': 2},
            'dishwasher': {'duration': 1.5, 'window': (7, 23), 'priority': 2},
            'ev_charger': {'duration': 4, 'window': (0, 24), 'priority': 1},
            'water_heater': {'duration': 3, 'window': (0, 24), 'priority': 3},
            'pool_pump': {'duration': 6, 'window': (8, 20), 'priority': 4},
            'dryer': {'duration': 1, 'window': (8, 21), 'priority': 2}
        }
        return constraints.get(appliance, {'duration': 1, 'window': (0, 24), 'priority': 3})
    
    def calculate_cost(self, schedule, consumption_forecast):
        total_cost = 0
        modified_consumption = consumption_forecast.copy()
        
        for appliance, start_time in schedule.items():
            profile = self.appliance_profiles[appliance]
            constraints = self.get_appliance_constraints(appliance)
            duration = int(constraints['duration'])
            
            for t in range(duration):
                hour = (start_time + t) % 24
                modified_consumption[hour] += profile['power']
                total_cost += modified_consumption[hour] * self.tariff_schedule[hour]
        
        peak_factor = np.max(modified_consumption) / np.mean(modified_consumption)
        total_cost *= (1 + 0.1 * (peak_factor - 2))
        
        return total_cost, modified_consumption
    
    def optimize_schedule(self, base_consumption_forecast, user_preferences=None):
        appliances = list(self.appliance_profiles.keys())
        n_appliances = len(appliances)
        
        def objective(x):
            schedule = {}
            for i, appliance in enumerate(appliances):
                schedule[appliance] = int(x[i])
            
            cost, _ = self.calculate_cost(schedule, base_consumption_forecast)
            
            if user_preferences:
                for appliance, preferred_time in user_preferences.items():
                    if appliance in schedule:
                        penalty = abs(schedule[appliance] - preferred_time) * 10
                        cost += penalty
            
            return cost
        
        bounds = []
        for appliance in appliances:
            constraints = self.get_appliance_constraints(appliance)
            bounds.append(constraints['window'])
        
        result = differential_evolution(
            objective,
            bounds,
            maxiter=100,
            popsize=15,
            seed=42
        )
        
        optimal_schedule = {}
        for i, appliance in enumerate(appliances):
            optimal_schedule[appliance] = int(result.x[i])
        
        optimal_cost, optimal_consumption = self.calculate_cost(
            optimal_schedule, 
            base_consumption_forecast
        )
        
        return {
            'schedule': optimal_schedule,
            'cost': optimal_cost,
            'consumption_profile': optimal_consumption,
            'savings': self._calculate_savings(base_consumption_forecast, optimal_consumption)
        }
    
    def _calculate_savings(self, base, optimized):
        base_cost = sum(base[i] * self.tariff_schedule[i] for i in range(24))
        optimized_cost = sum(optimized[i] * self.tariff_schedule[i] for i in range(24))
        savings_amount = base_cost - optimized_cost
        savings_percent = (savings_amount / base_cost) * 100 if base_cost > 0 else 0
        return {'amount': savings_amount, 'percent': savings_percent}
    
    def generate_recommendations(self, optimal_schedule):
        recommendations = []
        current_time = pd.Timestamp.now()
        
        for appliance, start_hour in optimal_schedule['schedule'].items():
            constraints = self.get_appliance_constraints(appliance)
            start_time = current_time.replace(hour=start_hour, minute=0, second=0)
            end_time = start_time + pd.Timedelta(hours=constraints['duration'])
            
            rec = {
                'appliance': appliance,
                'recommended_start': start_time.strftime('%H:%M'),
                'recommended_end': end_time.strftime('%H:%M'),
                'duration_hours': constraints['duration'],
                'priority': constraints['priority'],
                'estimated_cost': optimal_schedule['cost'] * (constraints['duration'] / 24),
                'tariff_type': 'off-peak' if self.tariff_schedule[start_hour] < np.mean(self.tariff_schedule) else 'peak'
            }
            recommendations.append(rec)
        
        recommendations.sort(key=lambda x: x['priority'])
        
        return recommendations

In [4]:
class AdvancedEnergyProcessor:
    def __init__(self, sequence_length=24, prediction_horizon=1):
        self.sequence_length = sequence_length
        self.prediction_horizon = prediction_horizon
        self.scaler = StandardScaler()
        self.target_scaler = MinMaxScaler()
        
    def create_advanced_features(self, data):
        data['hour'] = np.arange(len(data)) % 24
        data['day_of_week'] = (np.arange(len(data)) // 24) % 7
        data['is_weekend'] = data['day_of_week'].isin([5, 6]).astype(int)
        
        data['rolling_mean_6h'] = data['powerallphases'].rolling(6, center=True).mean()
        data['rolling_std_6h'] = data['powerallphases'].rolling(6, center=True).std()
        data['rolling_mean_24h'] = data['powerallphases'].rolling(24, center=True).mean()
        
        data['lag_1h'] = data['powerallphases'].shift(1)
        data['lag_24h'] = data['powerallphases'].shift(24)
        data['lag_168h'] = data['powerallphases'].shift(168)
        
        data['hour_sin'] = np.sin(2 * np.pi * data['hour'] / 24)
        data['hour_cos'] = np.cos(2 * np.pi * data['hour'] / 24)
        data['dow_sin'] = np.sin(2 * np.pi * data['day_of_week'] / 7)
        data['dow_cos'] = np.cos(2 * np.pi * data['day_of_week'] / 7)
        
        data = data.fillna(method='ffill').fillna(method='bfill')
        
        return data
    
    def create_weather_features(self, n_samples):
        np.random.seed(42)
        hours = np.arange(n_samples) % 24
        days = np.arange(n_samples) // 24
        
        temp_base = 20 + 10 * np.sin(2 * np.pi * days / 365)
        temperature = temp_base + 5 * np.sin(2 * np.pi * hours / 24) + np.random.normal(0, 2, n_samples)
        
        humidity = 60 + 20 * np.sin(2 * np.pi * hours / 24 + np.pi/4) + np.random.normal(0, 5, n_samples)
        humidity = np.clip(humidity, 30, 95)
        
        cloud_cover = 50 + 30 * np.sin(2 * np.pi * days / 30) + np.random.normal(0, 10, n_samples)
        cloud_cover = np.clip(cloud_cover, 0, 100)
        
        solar_radiation = np.maximum(0, 800 * np.sin(np.pi * np.clip((hours - 6) / 12, 0, 1)) * (1 - cloud_cover/100))
        
        return pd.DataFrame({
            'temperature': temperature,
            'humidity': humidity,
            'cloud_cover': cloud_cover,
            'solar_radiation': solar_radiation
        })
    
    def create_tariff_schedule(self):
        tariff = np.zeros(24)
        
        tariff[0:6] = 0.05
        tariff[6:9] = 0.12
        tariff[9:17] = 0.08
        tariff[17:21] = 0.15
        tariff[21:24] = 0.05
        
        return tariff

In [5]:
def train_enhanced_model(X_train, y_train, X_val, y_val, weather_train=None, weather_val=None,
                         epochs=150, batch_size=32, learning_rate=0.001):
    
    model = EnhancedGRANXModel(
        sequence_length=X_train.shape[1],
        n_features=X_train.shape[2],
        hidden_units=128,
        attention_heads=12,
        correction_rate=0.015
    )
    
    optimizer = keras.optimizers.AdamW(learning_rate=learning_rate, weight_decay=1e-5)
    model.compile(
        optimizer=optimizer,
        loss='huber',
        metrics=['mae', 'mse']
    )
    
    callbacks = [
        keras.callbacks.EarlyStopping(patience=30, restore_best_weights=True, monitor='val_loss'),
        keras.callbacks.ReduceLROnPlateau(patience=15, factor=0.5, min_lr=1e-7),
        keras.callbacks.ModelCheckpoint('granx_enhanced_best.h5', save_best_only=True)
    ]
    
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=batch_size,
        callbacks=callbacks,
        verbose=1
    )
    
    return model, history

def evaluate_enhanced_model(model, X_test, y_test, target_scaler):
    y_pred = model.predict(X_test)
    y_test_original = target_scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()
    y_pred_original = target_scaler.inverse_transform(y_pred.reshape(-1, 1)).flatten()

    mse = mean_squared_error(y_test_original, y_pred_original)
    mae = mean_absolute_error(y_test_original, y_pred_original)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test_original, y_pred_original)

    mape = np.mean(np.abs((y_test_original - y_pred_original) / y_test_original)) * 100
    
    print(f"Enhanced Model Performance:")
    print(f"RMSE: {rmse:.2f}")
    print(f"MAE: {mae:.2f}")
    print(f"MSE: {mse:.2f}")
    print(f"R²: {r2:.4f}")
    print(f"MAPE: {mape:.2f}%")
    
    return {
        'rmse': rmse, 'mae': mae, 'mse': mse, 'r2': r2, 'mape': mape,
        'y_true': y_test_original, 'y_pred': y_pred_original
    }

In [6]:
def plot_optimization_results(results, optimization_results):
    """Plot comprehensive optimization results"""
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    # Model Performance
    axes[0, 0].scatter(results['y_true'], results['y_pred'], alpha=0.6, color='blue')
    axes[0, 0].plot([results['y_true'].min(), results['y_true'].max()], 
                    [results['y_true'].min(), results['y_true'].max()], 'r--', lw=2)
    axes[0, 0].set_xlabel('Actual Energy (kWh)')
    axes[0, 0].set_ylabel('Predicted Energy (kWh)')
    axes[0, 0].set_title(f"Model Performance (R² = {results['r2']:.3f})")
    axes[0, 0].grid(True, alpha=0.3)
    
    # Cost Optimization
    if 'optimized_schedule' in optimization_results:
        schedule_data = optimization_results['optimized_schedule']
        hours = range(len(schedule_data))
        costs = [item['cost_per_hour'] for item in schedule_data]
        consumption = [item['energy_consumption'] for item in schedule_data]
        
        ax1 = axes[0, 1]
        ax2 = ax1.twinx()
        
        line1 = ax1.plot(hours, costs, 'g-', label='Cost per Hour', linewidth=2)
        line2 = ax2.plot(hours, consumption, 'b--', label='Energy Consumption', linewidth=2)
        
        ax1.set_xlabel('Hour of Day')
        ax1.set_ylabel('Cost ($)', color='g')
        ax2.set_ylabel('Energy (kWh)', color='b')
        ax1.set_title('Optimized Energy Schedule')
        
        lines = line1 + line2
        labels = [l.get_label() for l in lines]
        ax1.legend(lines, labels, loc='upper left')
        ax1.grid(True, alpha=0.3)
    
    # Time Series Comparison
    n_points = min(200, len(results['y_true']))
    axes[1, 0].plot(results['y_true'][:n_points], label='Actual', alpha=0.8, color='blue')
    axes[1, 0].plot(results['y_pred'][:n_points], label='Predicted', alpha=0.8, color='red')
    axes[1, 0].set_xlabel('Time Steps')
    axes[1, 0].set_ylabel('Energy Consumption (kWh)')
    axes[1, 0].set_title('Time Series Prediction Comparison')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # Error Distribution
    errors = results['y_true'] - results['y_pred']
    axes[1, 1].hist(errors, bins=30, alpha=0.7, color='purple', edgecolor='black')
    axes[1, 1].axvline(0, color='red', linestyle='--', linewidth=2)
    axes[1, 1].set_xlabel('Prediction Error (kWh)')
    axes[1, 1].set_ylabel('Frequency')
    axes[1, 1].set_title(f'Error Distribution (MAE = {results["mae"]:.2f})')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

In [ ]:
def main_enhanced():
    print("="*60)
    print("GRAN-X ENHANCED ENERGY OPTIMIZATION SYSTEM")
    print("="*60)
    
    processor = AdvancedEnergyProcessor(sequence_length=24, prediction_horizon=1)
    
    print("\n[1] Generating synthetic dataset with advanced features...")
    n_samples = 5000
    base_data = pd.DataFrame({
        'powerallphases': 500 + 200 * np.sin(2 * np.pi * np.arange(n_samples) % 24 / 24) + 
                         100 * np.sin(2 * np.pi * np.arange(n_samples) // 24 / 7) + 
                         50 * np.random.normal(0, 1, n_samples),
        'powerl1': 200 + 50 * np.random.normal(0, 1, n_samples),
        'powerl2': 150 + 40 * np.random.normal(0, 1, n_samples),
        'powerl3': 150 + 40 * np.random.normal(0, 1, n_samples),
        'currentneutral': 10 + 2 * np.random.normal(0, 1, n_samples),
        'currentl1': 4 + np.random.normal(0, 0.5, n_samples),
        'currentl2': 3 + np.random.normal(0, 0.5, n_samples),
        'currentl3': 3 + np.random.normal(0, 0.5, n_samples),
        'voltagel1': 230 + np.random.normal(0, 5, n_samples),
        'voltagel2': 230 + np.random.normal(0, 5, n_samples),
        'voltagel3': 230 + np.random.normal(0, 5, n_samples)
    })
    
    data = processor.create_advanced_features(base_data)
    weather_data = processor.create_weather_features(n_samples)
    data = pd.concat([data, weather_data], axis=1)
    
    print(f"Dataset shape: {data.shape}")
    print(f"Features: {list(data.columns)[:10]}...")
    
    X, y = [], []
    feature_cols = [col for col in data.columns if col not in ['powerallphases']]
    for i in range(processor.sequence_length, len(data) - processor.prediction_horizon):
        X.append(data[feature_cols].iloc[i-processor.sequence_length:i].values)
        y.append(data['powerallphases'].iloc[i+processor.prediction_horizon-1])
    
    X = np.array(X)
    y = np.array(y)
    
    X_scaled = processor.scaler.fit_transform(X.reshape(-1, X.shape[-1])).reshape(X.shape)
    y_scaled = processor.target_scaler.fit_transform(y.reshape(-1, 1)).flatten()
    
    split_idx = int(0.7 * len(X_scaled))
    val_idx = int(0.85 * len(X_scaled))
    
    X_train, y_train = X_scaled[:split_idx], y_scaled[:split_idx]
    X_val, y_val = X_scaled[split_idx:val_idx], y_scaled[split_idx:val_idx]
    X_test, y_test = X_scaled[val_idx:], y_scaled[val_idx:]
    
    print(f"\n[2] Data splits:")
    print(f"Training: {X_train.shape}")
    print(f"Validation: {X_val.shape}")
    print(f"Testing: {X_test.shape}")
    
    print("\n[3] Training Enhanced GRAN-X Model...")
    model, history = train_enhanced_model(
        X_train, y_train, X_val, y_val,
        epochs=50, batch_size=32, learning_rate=0.001
    )
    
    print("\n[4] Evaluating Model Performance...")
    results = evaluate_enhanced_model(model, X_test, y_test, processor.target_scaler)
    
    print("\n[5] Initializing Appliance Optimizer...")
    tariff_schedule = processor.create_tariff_schedule()
    
    appliance_profiles = {
        'washing_machine': {'power': 80, 'duration': 2},
        'dishwasher': {'power': 60, 'duration': 1.5},
        'ev_charger': {'power': 200, 'duration': 4},
        'water_heater': {'power': 120, 'duration': 3},
        'pool_pump': {'power': 50, 'duration': 6},
        'dryer': {'power': 100, 'duration': 1}
    }
    
    optimizer = ApplianceOptimizer(model, tariff_schedule, appliance_profiles)
    
    # Fixed forecast generation section
    print("\n[6] Generating 24-hour Forecast...")
    test_sequence = X_test[-1:].copy()
    forecast = []
    for i in range(24):
        pred = model.predict(test_sequence, verbose=0)
        forecast_value = processor.target_scaler.inverse_transform(pred.reshape(-1, 1))[0, 0]
        forecast.append(forecast_value)
        
    # Update sequence for next prediction
    test_sequence = np.roll(test_sequence, -1, axis=1)
    test_sequence[0, -1, 0] = pred[0, 0]  # Use normalized prediction

    print(f"24-hour forecast generated: {len(forecast)} values")
    
    forecast = np.array(forecast)
    
    print("\n[7] Optimizing Appliance Schedule...")
    user_preferences = {
        'washing_machine': 10,
        'ev_charger': 22
    }
    
    optimization_results = optimizer.optimize_schedule(forecast, user_preferences)
    
    print("\n" + "="*60)
    print("OPTIMIZATION RESULTS")
    print("="*60)
    
    print("\nOptimal Schedule:")
    for appliance, start_time in optimization_results['schedule'].items():
        print(f"  {appliance.replace('_', ' ').title()}: Start at {start_time:02d}:00")
    
    print(f"\nEstimated Daily Cost: ${optimization_results['cost']:.2f}")
    print(f"Savings: ${optimization_results['savings']['amount']:.2f} ({optimization_results['savings']['percent']:.1f}%)")
    
    print("\n[8] Generating Recommendations...")
    recommendations = optimizer.generate_recommendations(optimization_results)
    
    print("\nAPPLIANCE RECOMMENDATIONS:")
    print("-"*60)
    for rec in recommendations:
        print(f"\n{rec['appliance'].replace('_', ' ').title()}:")
        print(f"  Recommended Time: {rec['recommended_start']} - {rec['recommended_end']}")
        print(f"  Duration: {rec['duration_hours']} hours")
        print(f"  Priority: {'High' if rec['priority'] <= 2 else 'Medium' if rec['priority'] <= 3 else 'Low'}")
        print(f"  Tariff Period: {rec['tariff_type'].upper()}")
        print(f"  Estimated Cost: ${rec['estimated_cost']:.2f}")
    
    print("\n[9] Generating Visualization...")
    plot_optimization_results(results, optimization_results)
    
    print("\n" + "="*60)
    print("GRAN-X OPTIMIZATION COMPLETE")
    print("="*60)
    
    return model, results, optimization_results

model, results, optimization = main_enhanced()

GRAN-X ENHANCED ENERGY OPTIMIZATION SYSTEM

[1] Generating synthetic dataset with advanced features...
Dataset shape: (5000, 28)
Features: ['powerallphases', 'powerl1', 'powerl2', 'powerl3', 'currentneutral', 'currentl1', 'currentl2', 'currentl3', 'voltagel1', 'voltagel2']...

[2] Data splits:
Training: (3482, 24, 27)
Validation: (746, 24, 27)
Testing: (747, 24, 27)

[3] Training Enhanced GRAN-X Model...
Epoch 1/50
109/109 ━━━━━━━━━━━━━━━━━━━━ 0s 167ms/step - loss: 0.0381 - mae: 0.2132 - mse: 0.0761

109/109 ━━━━━━━━━━━━━━━━━━━━ 33s 196ms/step - loss: 0.0216 - mae: 0.1599 - mse: 0.0432 - val_loss: 0.0110 - val_mae: 0.1230 - val_mse: 0.0219 - learning_rate: 0.0010
Epoch 2/50
109/109 ━━━━━━━━━━━━━━━━━━━━ 0s 185ms/step - loss: 0.0119 - mae: 0.1228 - mse: 0.0237

109/109 ━━━━━━━━━━━━━━━━━━━━ 22s 198ms/step - loss: 0.0118 - mae: 0.1219 - mse: 0.0236 - val_loss: 0.0078 - val_mae: 0.1015 - val_mse: 0.0156 - learning_rate: 0.0010
Epoch 3/50
109/109 ━━━━━━━━━━━━━━━━━━━━ 0s 163ms/step - loss: 0.0112 - mae: 0.1199 - mse: 0.0224

109/109 ━━━━━━━━━━━━━━━━━━━━ 19s 176ms/step - loss: 0.0106 - mae: 0.1163 - mse: 0.0212 - val_loss: 0.0070 - val_mae: 0.0976 - val_mse: 0.0140 - learning_rate: 0.0010
Epoch 4/50
109/109 ━━━━━━━━━━━━━━━━━━━━ 0s 156ms/step - loss: 0.0098 - mae: 0.1127 - mse: 0.0195

109/109 ━━━━━━━━━━━━━━━━━━━━ 18s 168ms/step - loss: 0.0101 - mae: 0.1150 - mse: 0.0202 - val_loss: 0.0068 - val_mae: 0.0947 - val_mse: 0.0137 - learning_rate: 0.0010
Epoch 5/50
109/109 ━━━━━━━━━━━━━━━━━━━━ 17s 158ms/step - loss: 0.0094 - mae: 0.1102 - mse: 0.0187 - val_loss: 0.0073 - val_mae: 0.0982 - val_mse: 0.0145 - learning_rate: 0.0010
Epoch 6/50
109/109 ━━━━━━━━━━━━━━━━━━━━ 16s 143ms/step - loss: 0.0089 - mae: 0.1068 - mse: 0.0178 - val_loss: 0.0074 - val_mae: 0.0998 - val_mse: 0.0149 - learning_rate: 0.0010
Epoch 7/50
109/109 ━━━━━━━━━━━━━━━━━━━━ 0s 135ms/step - loss: 0.0085 - mae: 0.1056 - mse: 0.0171

...

109/109 ━━━━━━━━━━━━━━━━━━━━ 17s 151ms/step - loss: 0.0075 - mae: 0.0988 - mse: 0.0150 - val_loss: 0.0064 - val_mae: 0.0909 - val_mse: 0.0127 - learning_rate: 0.0010
Epoch 12/50
109/109 ━━━━━━━━━━━━━━━━━━━━ 0s 138ms/step - loss: 0.0073 - mae: 0.0967 - mse: 0.0146

109/109 ━━━━━━━━━━━━━━━━━━━━ 17s 152ms/step - loss: 0.0071 - mae: 0.0951 - mse: 0.0142 - val_loss: 0.0060 - val_mae: 0.0872 - val_mse: 0.0119 - learning_rate: 0.0010
Epoch 13/50
109/109 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step - loss: 0.0063 - mae: 0.0908 - mse: 0.0126

109/109 ━━━━━━━━━━━━━━━━━━━━ 16s 150ms/step - loss: 0.0062 - mae: 0.0896 - mse: 0.0125 - val_loss: 0.0054 - val_mae: 0.0816 - val_mse: 0.0108 - learning_rate: 0.0010
Epoch 14/50
109/109 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step - loss: 0.0061 - mae: 0.0875 - mse: 0.0122

109/109 ━━━━━━━━━━━━━━━━━━━━ 16s 149ms/step - loss: 0.0061 - mae: 0.0883 - mse: 0.0122 - val_loss: 0.0051 - val_mae: 0.0793 - val_mse: 0.0102 - learning_rate: 0.0010
Epoch 15/50
109/109 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step - loss: 0.0059 - mae: 0.0872 - mse: 0.0118

109/109 ━━━━━━━━━━━━━━━━━━━━ 16s 149ms/step - loss: 0.0058 - mae: 0.0861 - mse: 0.0117 - val_loss: 0.0046 - val_mae: 0.0762 - val_mse: 0.0093 - learning_rate: 0.0010
Epoch 16/50
109/109 ━━━━━━━━━━━━━━━━━━━━ 16s 143ms/step - loss: 0.0055 - mae: 0.0844 - mse: 0.0110 - val_loss: 0.0047 - val_mae: 0.0769 - val_mse: 0.0095 - learning_rate: 0.0010
Epoch 17/50
109/109 ━━━━━━━━━━━━━━━━━━━━ 16s 145ms/step - loss: 0.0053 - mae: 0.0818 - mse: 0.0106 - val_loss: 0.0054 - val_mae: 0.0831 - val_mse: 0.0109 - learning_rate: 0.0010
Epoch 18/50
109/109 ━━━━━━━━━━━━━━━━━━━━ 16s 146ms/step - loss: 0.0050 - mae: 0.0796 - mse: 0.0099 - val_loss: 0.0050 - val_mae: 0.0784 - val_mse: 0.0099 - learning_rate: 0.0010
Epoch 19/50
109/109 ━━━━━━━━━━━━━━━━━━━━ 16s 147ms/step - loss: 0.0051 - mae: 0.0803 - mse: 0.0103 - val_loss: 0.0051 - val_mae: 0.0779 - val_mse: 0.0101 - learning_rate: 0.0010
Epoch 20/50
109/109 ━━━━━━━━━━━━━━━━━━━━ 17s 158ms/step - loss: 0.0047 - mae: 0.0767 - mse: 0.0095 - val_loss: 0.0049 - va